<a href="https://colab.research.google.com/github/ajakhmol/CNN/blob/main/SummaryCreater.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# prompt: write a iterator on the criteria_list list and pass the value of the list in the generate_response function
import pandas as pd
from IPython.display import display, HTML
import os, json, ast
import openai
from tenacity import retry, wait_random_exponential, stop_after_attempt
from google.colab import drive
# Set the display width to control the output width
# pd.set_option('display.width', 500)
from io import StringIO

from tenacity import retry, wait_random_exponential, stop_after_attempt

In [ ]:
column_list = [
"Title",
"Author",
"Year of publication",
"Keywords",
"Objectives",
"Domain or Application Area",
"Type of Research",
"Key methods",
"AI/ML Model Created with Accuracy and Evaluation Metrics Used"
"Baseline Models or Comparison Models",
"Major findings",
"Important conclusions",
"Data Source on which Model cerated",
"Dataset Size and Characteristics",
"Software/Tools/Frameworks Used",
"Ethical or Societal Implications",
"Code or Repository Availability",
"Limitations or future directions"
]



In [ ]:

# Importing the necessary library for mounting Google Drive

# Mounting Google Drive to the Colab environment
drive.mount('/content/drive', force_remount=True)
# Read the API key from the text file and strip any leading or trailing whitespace
with open("/content/drive/My Drive/Gen_AI/OPENAI_API_Key.txt", "r") as f:
    api_key = f.read().strip()

# Set the API key for OpenAI
openai.api_key = api_key

Mounted at /content/drive


In [ ]:
# @retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(6))
# def generate_ai_response_subtopic(df_csv):
#     """
#     Generate a response using gpt-4o-mini ChatCompletion.  Returns a CSV string.
#     """
#     system_content="You are a helpful AI assistant in the Explainable Articifal Intellengnce(XAI) Technology consulting domain, specialized in providing accurate answers. Respond with ONLY a CSV string, with the first row being the header. Do not include any text before or after the CSV data."

#     user_content= f"""
#         Please summarize the following research paper
#                   Title of the document if not avaliable then say NA
#                   Author of the document if not avaliable then say NA
#                   Year of publication of the document if not avaliable then say NA
#                   The main objectives or research questions of the document Not more than 50 words
#                   Key methods or approach used of the document Not more than 100 words
#                   Major findings or results of the document Not more than 50 words
#                   Important conclusions  of the document Not more than 50 words
#                   Any limitations or future directions mentioned of the document Not more than 50 words
#               """

#     messages = [
#         {"role": "system", "content": system_content },
#         {"role": "user", "content": user_content },
#     ]

#     response = openai.chat.completions.create(
#         model="gpt-4o-mini",
#         messages=messages
#     )
#     content = response.choices[0].message.content
#     return content # Return the raw CSV string

In [ ]:
import fitz  # PyMuPDF

def extract_text_from_pdf(pdf_path):
    """Extracts text from a PDF file."""
    try:
        doc = fitz.open(pdf_path)
        text = ""
        for page in doc:
            text += page.get_text()
        return text
    except Exception as e:
        print(f"Error reading PDF {pdf_path}: {e}")
        return None

@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(6))
def generate_ai_response_summary(text):
    """
    Generate a response using gpt-4o-mini ChatCompletion. Returns a dictionary.
    """
    system_content="""You are a helpful AI assistant in the Explainable Articifal Intellengnce(XAI) Technology
                      consulting domain, specialized in providing accurate answers. Respond with ONLY a json object, with the keys
                      as specified in the user prompt. Do not include any text before or after the json data."""

    user_content= f"""
        Please summarize the following research paper, using the following keys:
                  "Title"
                  "Author"
                  "Year of publication"
                  "Keywords"
                  "Objectives",
                  "Domain or Application Area"
                  "Type of Research"
                  "Key methods"
                  "AI/ML Model Created with Accuracy and Evaluation Metrics Used"
                  "Baseline Models or Comparison Models"
                  "Major findings"
                  "Important conclusions"
                  "Data Source on which Model cerated"
                  "Dataset Size and Characteristics"
                  "Software/Tools/Frameworks Used"
                  "Ethical or Societal Implications"
                  "Code or Repository Availability"
                  "Limitations or future directions"

                  Conditions for these keys:
                  Title of the document if not avaliable then say NA
                  Author of the document if not avaliable then say NA
                  Year of publication of the document if not avaliable then say NA
                  Keywords or Tags, Helps identify the core themes or domains (e.g., XAI, NLP, SHAP, fairness, deep learning, etc.)
                  The main objectives or research questions of the document Not more than 100 words
                  Domain or Application Area, What field does the paper apply to? (e.g., healthcare, finance, education, climate science, etc.)
                  Type of Research, Is it theoretical, empirical, experimental, review, or applied?
                   Key methods or approach used of the document Not more than 150 words
                  AI/ML Model and Evaluation Metrics Used, Accuracy, F1-score, AUC, RMSE, MAE, Precision, Recall, etc. If not mentioned, say NA.
                  Major findings or results of the document Not more than 100 words
                  Important conclusions  of the document Not more than 100 words
                  Data Source on which Model cerated, give the details how I can get that data if not avaliable then say NA
                  Dataset Size and Characteristics ,Number of samples, features, class balance, etc. This helps evaluate model robustness. If not available, say NA.
                  Software/Tools/Frameworks Used, List tools like Python, R, TensorFlow, PyTorch, Scikit-learn, etc., if mentioned.
                  Baseline Models or Comparison Models, Did the authors compare their model with others? Mention the baselines and whether their model outperformed them.
                  Ethical or Societal Implications, If the paper discusses bias, fairness, privacy, or any ethical concerns.
                  Code or Repository Availability,  Provide a GitHub or other repo link if shared. If not available, say NA.
                  Any limitations or future directions mentioned of the document Not more than 100 words


        Text to summarize:
        {text}
              """

    messages = [
        {"role": "system", "content": system_content },
        {"role": "user", "content": user_content },
    ]

    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )
    content = response.choices[0].message.content
    print(content)
    return json.loads(content)

def process_pdfs(pdf_directory, output_excel_path):
    """
    Processes all PDFs in a directory, generates summaries, and saves them to an Excel file.
    """
    pdf_files = [f for f in os.listdir(pdf_directory) if f.endswith(".pdf")]
    all_summaries = []

    for pdf_file in pdf_files:
        pdf_path = os.path.join(pdf_directory, pdf_file)
        print(f"Processing {pdf_file}...")
        text = extract_text_from_pdf(pdf_path)
        print(f"Text extracted from {pdf_file}")
        if text:
            summary_json = generate_ai_response_summary(text)
            all_summaries.append(summary_json)

    df = pd.DataFrame(all_summaries, columns=column_list)
    df.to_excel(output_excel_path, index=False)
    print(f"Summaries saved to {output_excel_path}")

In [ ]:
# pdf_directory = "/content/drive/My Drive/Gen_AI/ResearchPaper/In"
# output_excel_path = "/content/drive/My Drive/Gen_AI/ResearchPaper/Out/pdf_summaries.xlsx"
# process_pdfs(pdf_directory, output_excel_path)

Processing Understanding_Model_Predictions_A_Comparative_Anal.pdf...
Text extracted from Understanding_Model_Predictions_A_Comparative_Anal.pdf
Summaries saved to /content/drive/My Drive/Gen_AI/ResearchPaper/Out/pdf_summaries.xlsx
